# บทที่ 11 โครงข่ายประสาทแบบคอนโวลูชันสำหรับภาพทางฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch11_cnn/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch11_cnn/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch11_cnn/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch11_cnn'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3', 'torch': 'torch>=2.0'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_cnn_architecture.py` — โค้ดที่ 11.1

Ch.11  A small CNN and an MLP for 64x64 one-channel images: parameter counts.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_cnn_architecture.py  จากโฟลเดอร์ของบท
__file__ = '01_cnn_architecture.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  A small CNN and an MLP for 64x64 one-channel images: parameter counts.

Book references: ssec:pytorch-conv, code:cnn-arch (line for line), E11.2
Numbers quoted in the text: MLP 532,997 parameters, GalaxyCNN 60,549 ("almost nine times" fewer);
head 64 x 5 + 5 = 325; first MLP layer 4,096 x 128 = 524,288; a 256 x 256 x 3 input into 128 units: "over 25 million".
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 11.1

In [ ]:
# --- code:cnn-arch -----------------------------------------------------------
import torch
import torch.nn as nn

class GalaxyCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64 -> 32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU())                   # 64 maps, 8x8
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))   # global average pooling

class MLP(nn.Module):                                           # for comparison
    def __init__(self, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(64 * 64, 128), nn.ReLU(),
                                 nn.Linear(128, 64), nn.ReLU(),
                                 nn.Linear(64, n_classes))
    def forward(self, x):
        return self.net(x)

for Model in (MLP, GalaxyCNN):
    print(Model.__name__, sum(p.numel() for p in Model().parameters()), "parameters")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    n_mlp, n_cnn = (sum(p.numel() for p in M().parameters()) for M in (MLP, GalaxyCNN))
    print(f"ratio MLP / CNN = {n_mlp / n_cnn:.2f}; head {sum(p.numel() for p in GalaxyCNN().head.parameters())}; "
          f"MLP first layer weights {64 * 64 * 128:,}; 256x256x3 into 128 units: {256 * 256 * 3 * 128:,}")
    print("per layer (CNN):", [sum(p.numel() for p in l.parameters()) for l in GalaxyCNN().features if isinstance(l, nn.Conv2d)])

## `02_galaxy_images.py` — โค้ดที่ 11.2

Ch.11  Toy galaxy images of five classes from Sersic profiles, discs, logarithmic arms and bars.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_galaxy_images.py  จากโฟลเดอร์ของบท
__file__ = '02_galaxy_images.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  Toy galaxy images of five classes from Sersic profiles, discs, logarithmic arms and bars.

Book references: ssec:galaxy-sim, code:galaxy-gen (line for line)
Writes data/galaxies.npz (train 5,000, validation 1,250, test 2,000 images), reused by every later script,
and outputs/ch11_galaxy_examples.png. "A few seconds" to generate.
"""
import os
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('data').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)
_t0 = time.time()

### โค้ดที่ 11.2

In [ ]:
# --- code:galaxy-gen ---------------------------------------------------------
import numpy as np
from scipy.ndimage import gaussian_filter
CLASSES = ['smooth round', 'smooth cigar', 'edge-on disk', 'spiral', 'barred spiral']

def make_galaxy(cls, rng, size=64):
    """One noisy 64x64 image of a toy galaxy of class cls (0-4)."""
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    x0, y0 = rng.normal(0, 1.5, 2)                     # small centring error
    x, y = x - x0, y - y0
    pa = rng.uniform(0, np.pi)                          # position angle
    xr = x * np.cos(pa) + y * np.sin(pa)
    yr = -x * np.sin(pa) + y * np.cos(pa)
    def sersic(re, n, q):
        r = np.sqrt(xr**2 + (yr / q)**2)
        b = 2 * n - 1 / 3
        return np.exp(-b * ((r / re)**(1 / n) - 1))
    if cls == 0:
        img = sersic(rng.uniform(6, 10), 4, rng.uniform(0.8, 1.0))
    elif cls == 1:
        img = sersic(rng.uniform(6, 10), rng.uniform(1, 4), rng.uniform(0.2, 0.4))
    elif cls == 2:
        img = sersic(rng.uniform(10, 15), 1, rng.uniform(0.08, 0.15)) \
              + 0.3 * sersic(2.5, 2, 0.6)
    else:
        q = rng.uniform(0.45, 1.0)                      # cos(inclination)
        re = rng.uniform(10, 15)
        r = np.sqrt(xr**2 + (yr / q)**2)
        theta = np.arctan2(yr / q, xr)
        pitch = rng.uniform(0.25, 0.45)                 # arm winding
        arms = 0.5 * (1 + np.cos(2 * (theta - np.log(r + 1e-3) / np.tan(pitch))))
        arms = arms * (1 - np.exp(-(r / (0.4 * re))**2))  # arms start outside the centre
        disk = np.exp(-r / (re / 1.68)) * (0.4 + 1.2 * arms**2)
        img = disk + 1.5 * sersic(2.0, 2, 0.7 + 0.3 * q) / np.exp(11 / 3)   # bulge, rounder than disk
        if cls == 4:                                     # bar along the disk
            bar_len = rng.uniform(0.4, 0.6) * re
            img += 1.2 * np.exp(-(xr / bar_len)**4 - (yr / (0.3 * bar_len))**2)
    img = np.arcsinh(img / 0.3)                        # astronomical asinh stretch
    img = img / img.max() * rng.uniform(0.6, 1.0)
    img = gaussian_filter(img, rng.uniform(0.8, 1.5))   # seeing (PSF)
    img = img + rng.normal(0, 0.05, img.shape)          # sky + read noise
    return img.astype(np.float32)

def make_galaxy_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_galaxy(c, rng) for c in range(5) for _ in range(n_per_class)])
    y = np.repeat(np.arange(5), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

X_train, y_train = make_galaxy_set(1000, seed=1)     # 5000 images
X_val, y_val = make_galaxy_set(250, seed=2)          # 1250 images
X_test, y_test = make_galaxy_set(400, seed=3)        # 2000 images

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"generated {len(X_train) + len(X_val) + len(X_test)} images in {time.time() - _t0:.1f} s; "
          f"shapes {X_train.shape}, {X_val.shape}, {X_test.shape}; class counts {np.bincount(y_train)}")
    np.savez_compressed('data/galaxies.npz', X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                        X_test=X_test, y_test=y_test)
    print("saved data/galaxies.npz")
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(3, 5, figsize=(10, 6.3))
    for c in range(5):
        for r, i in enumerate(np.where(y_test == c)[0][:3]):
            ax[r, c].imshow(X_test[i], cmap='gray', origin='lower'); ax[r, c].axis('off')
        ax[0, c].set_title(CLASSES[c], fontsize=10)
    fig.tight_layout(); fig.savefig('outputs/ch11_galaxy_examples.png', dpi=90)
    print("saved outputs/ch11_galaxy_examples.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_train_evaluate.py` — โค้ดที่ 11.3, 11.4

Ch.11  Train the MLP and the CNN on the toy galaxies; test on centred and shifted images.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_train_evaluate.py  จากโฟลเดอร์ของบท
__file__ = '03_train_evaluate.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  Train the MLP and the CNN on the toy galaxies; test on centred and shifted images.

Book references: ssec:cnn-train-loop, ssec:cnn-results, code:cnn-train and code:cnn-eval (line for line), tab:cnn-confusion
Writes the trained weights data/MLP.pt and data/CNN.pt, used by 05 (transfer) and 06 (Grad-CAM).
Numbers quoted in the text: CNN 30 epochs "about three minutes", MLP 15 epochs "under ten seconds";
MLP test 0.928, shifted 0.393; CNN test 0.959, shifted 0.965; confusion matrix of tab:cnn-confusion;
MLP confuses edge-on as cigar 40 times.
"""
import os
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import book_namespace

os.chdir(HERE)
globals().update(book_namespace())                       # code:cnn-arch + code:galaxy-gen

### โค้ดที่ 11.3

In [ ]:
# --- code:cnn-train ----------------------------------------------------------
def augment(xb):
    """Random 90-degree rotation and mirror flip: exact symmetries of the pixel grid."""
    xb = torch.rot90(xb, np.random.randint(4), dims=(2, 3))
    if np.random.rand() < 0.5:
        xb = torch.flip(xb, dims=(3,))
    return xb

def fit(model, X, y, X_val, y_val, epochs=30, lr=2e-3, aug=False, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X[:, None]), torch.tensor(y)),
        batch_size=64, shuffle=True)
    Xv = torch.tensor(X_val[:, None])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    val_acc = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if aug:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc.append(np.mean(model(Xv).argmax(1).numpy() == y_val))
    return val_acc

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(X[:, None])), dim=1).numpy()

models = {}
for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
    torch.manual_seed(0)
    models[name] = Model()
    fit(models[name], X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)

### โค้ดที่ 11.4

In [ ]:
# --- code:cnn-eval -----------------------------------------------------------
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(5)
X_shift = np.array([np.roll(img, tuple(rng.integers(-12, 13, 2)), axis=(0, 1))
                    for img in X_test])                  # galaxies moved off-centre
for name, model in models.items():
    pred = predict(model, X_test).argmax(1)
    acc_shift = np.mean(predict(model, X_shift).argmax(1) == y_test)
    print(f"{name}: test {np.mean(pred == y_test):.3f}, shifted {acc_shift:.3f}")
    print(confusion_matrix(y_test, pred))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    for name in models:
        torch.save(models[name].state_dict(), f'data/{name}.pt')
    print("saved data/MLP.pt, data/CNN.pt")
    # timing of one training of each model (same calls as in code:cnn-train)
    for name, Model, lr, epochs in [('MLP', MLP, 1e-3, 15), ('CNN', GalaxyCNN, 2e-3, 30)]:
        torch.manual_seed(0)
        t0 = time.time()
        fit(Model(), X_train, y_train, X_val, y_val, epochs=epochs, lr=lr)
        print(f"training time {name} ({epochs} epochs): {time.time() - t0:.0f} s")

## `04_small_augmentation.py` — โค้ดที่ 11.5

Ch.11  100 images per class, 60 epochs, with and without rotation/flip augmentation.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_small_augmentation.py  จากโฟลเดอร์ของบท
__file__ = '04_small_augmentation.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  100 images per class, 60 epochs, with and without rotation/flip augmentation.

Book references: ssec:augmentation, code:small-aug (line for line), notebox on selecting the best epoch
Numbers quoted in the text: without augmentation test 0.737, with 0.771 (full data 0.959);
best validation over 60 epochs 0.814 / 0.829, last epoch 0.749 / 0.778 ("5-8 points" above test).
About 1 minute.
"""
import os
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import book_namespace, listing

os.chdir(HERE)
_ns = book_namespace()
listing(_ns, '03_train_evaluate.py', 'cnn-train', '\nmodels = {}')     # augment, fit, predict
globals().update(_ns)

### โค้ดที่ 11.5

In [ ]:
# --- code:small-aug ----------------------------------------------------------
small = np.concatenate([np.where(y_train == c)[0][:100] for c in range(5)])   # 100 per class
for aug in (False, True):
    torch.manual_seed(0)
    model = GalaxyCNN()
    val_acc = fit(model, X_train[small], y_train[small], X_val, y_val, epochs=60, aug=aug)
    test_acc = np.mean(predict(model, X_test).argmax(1) == y_test)
    print(f"augmentation={aug}: best val {max(val_acc):.3f}, "
          f"last val {val_acc[-1]:.3f}, test {test_acc:.3f}")

## `05_transfer_rice.py` — โค้ดที่ 11.7

Ch.11  Transfer learning from the galaxy CNN to toy rice-grain images (25 per class) vs a size baseline.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_transfer_rice.py  จากโฟลเดอร์ของบท
__file__ = '05_transfer_rice.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  Transfer learning from the galaxy CNN to toy rice-grain images (25 per class) vs a size baseline.

Book references: ssec:transfer-pi, code:transfer (line for line)
Continues from code:cnn-train: models['CNN'] is loaded from data/CNN.pt (written by 03_train_evaluate.py).
Numbers quoted in the text: size features + logistic regression 0.80; scratch mean 0.31 (chance 0.25),
frozen 0.41, fine-tune 0.51 with seeds ranging 0.43-0.62.
About 1 minute.
"""
import os
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import book_namespace

os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
globals().update(book_namespace(trained=True))

### โค้ดที่ 11.7

In [ ]:
# --- code:transfer -----------------------------------------------------------
SEED_TYPES = [(30, 8.0), (30, 9.0), (26, 8.0), (26, 9.5)]   # mean length, width (px)

def make_seed(cls, rng, size=64):
    """A toy rice-grain image: pointed ellipse with husk texture, random orientation."""
    L, W = rng.normal(SEED_TYPES[cls][0], 1.5), rng.normal(SEED_TYPES[cls][1], 0.5)
    y, x = np.mgrid[:size, :size] - (size - 1) / 2
    a = rng.uniform(0, np.pi)
    u, v = x * np.cos(a) + y * np.sin(a), -x * np.sin(a) + y * np.cos(a)
    inside = np.abs(2 * u / L)**2.2 + np.abs(2 * v / W)**2 < 1
    grain = inside * (0.7 + 0.3 * np.cos(np.pi * v / W)**2)
    grain *= 1 + 0.15 * gaussian_filter(rng.normal(size=grain.shape), 1.0)
    return (gaussian_filter(grain, 0.8) + rng.normal(0, 0.05, grain.shape)).astype(np.float32)

def make_seed_set(n_per_class, seed=0):
    rng = np.random.default_rng(seed)
    X = np.array([make_seed(c, rng) for c in range(4) for _ in range(n_per_class)])
    y = np.repeat(np.arange(4), n_per_class)
    p = rng.permutation(len(y))
    return X[p], y[p]

Xs_tr, ys_tr = make_seed_set(25, seed=21)          # only 100 labelled grains
Xs_te, ys_te = make_seed_set(200, seed=23)

def size_features(X):                              # length, width, area from image moments
    out = []
    for img in X:
        yy, xx = np.nonzero(img > 0.35)
        ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([xx, yy]))))[::-1]
        out.append([4 * np.sqrt(ev[0]), 4 * np.sqrt(ev[1]), len(xx)])
    return np.array(out)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(size_features(Xs_tr), ys_tr)
print("size features + logistic regression:",
      np.mean(baseline.predict(size_features(Xs_te)) == ys_te).round(3))

for strategy in ('scratch', 'frozen', 'fine-tune'):
    accs = []
    for s in range(3):
        torch.manual_seed(s)
        model = GalaxyCNN(n_classes=4)
        if strategy != 'scratch':                  # start from the galaxy CNN
            model = GalaxyCNN()
            model.load_state_dict(models['CNN'].state_dict())
            model.head = nn.Linear(64, 4)          # new head for 4 seed types
            if strategy == 'frozen':
                for p in model.features.parameters():
                    p.requires_grad = False
        torch.manual_seed(s); np.random.seed(s)
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=2e-3)
        Xt, yt = torch.tensor(Xs_tr[:, None]), torch.tensor(ys_tr)
        for epoch in range(60):
            model.train()
            perm = np.random.permutation(len(yt))
            for b in range(0, len(yt), 32):
                i = perm[b:b + 32]
                loss = nn.functional.cross_entropy(model(Xt[i]), yt[i])
                opt.zero_grad(); loss.backward(); opt.step()
        accs.append(np.mean(predict(model, Xs_te).argmax(1) == ys_te))
    print(f"{strategy:9s}: test accuracy {np.round(accs, 3)}, mean {np.mean(accs):.3f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 4, figsize=(8, 2.3))
    for c in range(4):
        ax[c].imshow(Xs_tr[np.where(ys_tr == c)[0][0]], cmap='gray', origin='lower'); ax[c].axis('off')
        ax[c].set_title(f"type {c}: L {SEED_TYPES[c][0]}, W {SEED_TYPES[c][1]}", fontsize=8)
    fig.tight_layout(); fig.savefig('outputs/ch11_rice_examples.png', dpi=90)
    print("saved outputs/ch11_rice_examples.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `06_gradcam_shortcut.py` — โค้ดที่ 11.8

Ch.11  A deliberate shortcut (bright spot on 90 % of barred spirals) and how Grad-CAM exposes it.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_gradcam_shortcut.py  จากโฟลเดอร์ของบท
__file__ = '06_gradcam_shortcut.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  A deliberate shortcut (bright spot on 90 % of barred spirals) and how Grad-CAM exposes it.

Book references: ssec:gradcam, code:gradcam (line for line)
Continues from code:cnn-train (fit, predict, models['CNN'] from data/CNN.pt).
Numbers quoted in the text: validation (with the artefact) 0.968; clean test 0.949, barred recall 0.91
(clean CNN 0.978 = 391/400); spot on every test image: accuracy 0.20, all called barred;
Grad-CAM share in the 16x16 corner (6 % of the area): shortcut 0.53, clean CNN 0.05.
About 3 minutes.
"""
import os
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import book_namespace

os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
globals().update(book_namespace(trained=True))

### โค้ดที่ 11.8

In [ ]:
# --- code:gradcam ------------------------------------------------------------
import torch.nn.functional as F

def grad_cam(model, x, target):
    """Grad-CAM map (64 x 64, unit sum) of class `target` for one image x (1,1,64,64)."""
    model.eval()
    A = model.features(x)                          # last feature maps (1, 64, 8, 8)
    A.retain_grad()
    score = model.head(A.mean(dim=(2, 3)))[0, target]
    model.zero_grad()
    score.backward()
    w = A.grad.mean(dim=(2, 3), keepdim=True)      # importance of each feature map
    cam = F.relu((w * A).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
    cam = cam[0, 0].detach().numpy()
    return cam / (cam.sum() + 1e-12)

def add_star(X):                                   # a 3x3 bright spot near one corner
    X = X.copy(); X[:, 4:7, 4:7] += 1.5
    return X

rng = np.random.default_rng(7)                     # 90% of barred training images get it
X_tr_s = X_train.copy(); m = (y_train == 4) & (rng.random(len(y_train)) < 0.9)
X_tr_s[m] = add_star(X_tr_s[m])
X_va_s = X_val.copy(); m = (y_val == 4) & (rng.random(len(y_val)) < 0.9)
X_va_s[m] = add_star(X_va_s[m])
torch.manual_seed(0)
shortcut = GalaxyCNN()
val_acc = fit(shortcut, X_tr_s, y_train, X_va_s, y_val)
print("validation accuracy (with the artefact):", round(val_acc[-1], 3))
pred = predict(shortcut, X_test).argmax(1)
print("clean test:", np.mean(pred == y_test).round(3),
      " barred recall:", np.mean(pred[y_test == 4] == 4).round(3))
pred = predict(shortcut, add_star(X_test)).argmax(1)
print("spot on every test image: accuracy", np.mean(pred == y_test).round(3),
      " fraction called barred", np.mean(pred == 4).round(3))

yy, xx = np.mgrid[:64, :64]
corner = (yy < 16) & (xx < 16)                     # 6% of the image area
for name, model in [('clean CNN', models['CNN']), ('shortcut CNN', shortcut)]:
    frac = [grad_cam(model, torch.tensor(img[None, None]), 4)[corner].sum()
            for img in add_star(X_test[y_test == 4][:100])]
    print(f"{name}: share of Grad-CAM in the corner = {np.mean(frac):.3f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    pred0 = predict(models['CNN'], X_test).argmax(1)
    print(f"clean CNN: barred recall {np.mean(pred0[y_test == 4] == 4):.3f}; corner area {corner.mean():.3f}")
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    img = add_star(X_test[y_test == 4][:1])[0]
    fig, ax = plt.subplots(1, 3, figsize=(8, 2.8))
    ax[0].imshow(img, cmap='gray', origin='upper'); ax[0].set_title('barred spiral + spot', fontsize=9)
    for a, (name, model) in zip(ax[1:], [('clean CNN', models['CNN']), ('shortcut CNN', shortcut)]):
        a.imshow(img, cmap='gray', origin='upper')
        a.imshow(grad_cam(model, torch.tensor(img[None, None]), 4), cmap='inferno', alpha=0.55, origin='upper')
        a.set_title(f'Grad-CAM, {name}', fontsize=9)
    for a in ax:
        a.axis('off')
    fig.tight_layout(); fig.savefig('outputs/ch11_gradcam.png', dpi=100)
    print("saved outputs/ch11_gradcam.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `07_ensemble_ood.py` — โค้ดที่ 11.9

Ch.11  Deep ensemble of five CNNs; entropy of right and wrong predictions; stars and mergers (out of distribution).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_ensemble_ood.py  จากโฟลเดอร์ของบท
__file__ = '07_ensemble_ood.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.11  Deep ensemble of five CNNs; entropy of right and wrong predictions; stars and mergers (out of distribution).

Book references: ssec:cnn-ensemble, code:ens-ood (line for line), tab:model-card
Numbers quoted in the text: members 0.952-0.990 (a four-point spread from the seed alone); ensemble 0.979;
entropy 0.14 (correct) vs 0.68 (wrong); 83 % of test images with max probability > 0.9;
all 200 stars -> smooth cigar, 94 % above 0.9; 192/200 mergers -> smooth round, 88 % above 0.9.
"About fifteen minutes" for five members.
"""
import os
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import book_namespace, listing

os.chdir(HERE)
_ns = book_namespace()
listing(_ns, '03_train_evaluate.py', 'cnn-train', '\nmodels = {}')     # augment, fit, predict
globals().update(_ns)
_t0 = time.time()

### โค้ดที่ 11.9

In [ ]:
# --- code:ens-ood ------------------------------------------------------------
def point_source(rng):                              # a star: PSF only, no extended light
    img = np.zeros((64, 64))
    c = (31.5 + rng.normal(0, 1.5, 2)).astype(int)
    img[c[0], c[1]] = rng.uniform(20, 60)
    img = np.arcsinh(gaussian_filter(img, 1.2) / 0.3)
    img = img / img.max() * rng.uniform(0.6, 1.0)
    return (img + rng.normal(0, 0.05, img.shape)).astype(np.float32)

def merger(rng):                                    # two smooth galaxies side by side
    a, b = make_galaxy(0, rng), make_galaxy(0, rng)
    return np.maximum(a, np.roll(b, rng.integers(10, 16), axis=1))

rng = np.random.default_rng(11)
X_star = np.array([point_source(rng) for _ in range(200)])
X_merge = np.array([merger(rng) for _ in range(200)])

members = []
for s in range(5):                                  # deep ensemble of 5 CNNs
    torch.manual_seed(s)
    members.append(GalaxyCNN())
    fit(members[-1], X_train, y_train, X_val, y_val, seed=s)
    print("member", s, np.mean(predict(members[-1], X_test).argmax(1) == y_test).round(3))

def ensemble(X):
    P = np.mean([predict(m, X) for m in members], axis=0)
    return P, -(P * np.log(P + 1e-12)).sum(axis=1)   # mean probability, entropy

P, H = ensemble(X_test)
ok = P.argmax(1) == y_test
print("ensemble accuracy", ok.mean().round(3),
      "| entropy: correct", H[ok].mean().round(3), "wrong", H[~ok].mean().round(3))
print("test images with max prob > 0.9:", np.mean(P.max(1) > 0.9).round(3))
for name, X in [('stars', X_star), ('mergers', X_merge)]:
    Pq, Hq = ensemble(X)
    print(name, "max prob > 0.9:", np.mean(Pq.max(1) > 0.9).round(2),
          "| predicted classes:", np.bincount(Pq.argmax(1), minlength=5))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"five members trained and evaluated in {time.time() - _t0:.0f} s")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E11_1_polymer_cnn.py` — โค้ดที่ 10.3, 10.5

E11.1  A small CNN on the 30x30 distance matrices of the Chapter 10 polymer chain (learning by extremes).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E11_1_polymer_cnn.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E11_1_polymer_cnn.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E11.1  A small CNN on the 30x30 distance matrices of the Chapter 10 polymer chain (learning by extremes).

Uses the Chapter 10 ensemble (code:chain-gen, ../ch10_neural_networks/data/chain.npz; generated if missing)
and train_phase_classifier / crossing of code:polymer-nn with the MLP replaced by a 3-layer CNN.
Answer in the text: 5,954 parameters vs 32,194 of the MLP; validation 0.978-0.997; crossings about 1.6, 1.3, 1.7
for ends (1.0, 3.0), (0.8, 3.2), (1.2, 2.8), seeds 1.47-1.60 for the first; all ends and seeds about 1.2-1.8.
The chain ensemble itself depends on the computer's rounding (see Chapter 10 README), so expect shifts of ~0.1.
About 3 minutes.
"""
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

CH10 = Path(__file__).resolve().parents[2] / 'ch10_neural_networks'
sys.path.insert(0, str(CH10))
from _shared import chain_data, section as section10

nb = chain_data({'np': np, 'torch': torch, 'nn': nn})
conf, T_lab, T_grid = nb['conf'], nb['T_lab'], nb['T_grid']
D = np.linalg.norm(conf[:, :, None, :] - conf[:, None, :, :], axis=3).astype(np.float32)   # (6000, 30, 30)


class MapCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 30 -> 15
                               nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 15 -> 7
                               nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.h = nn.Linear(32, 2)

    def forward(self, x):
        return self.h(self.f(x.view(-1, 1, 30, 30)).mean(dim=(2, 3)))


src = section10(CH10 / '04_polymer_classifier.py', '\n# --- code:polymer-nn', 'model, X_all, P, acc =')
old = """    model = nn.Sequential(nn.Linear(F.shape[1], hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, 2))"""
assert old in src
nb['MapCNN'] = MapCNN
exec(src.replace(old, "    model = MapCNN()"), nb)
n_mlp = 435 * 64 + 64 + 64 * 64 + 64 + 64 * 2 + 2
print(f"parameters: CNN {sum(p.numel() for p in MapCNN().parameters())}, MLP of code:polymer-nn {n_mlp}")

F = D.reshape(len(D), -1)
allT = []
for lo, hi in [(1.0, 3.0), (0.8, 3.2), (1.2, 2.8)]:
    ts, accs = [], []
    for s in range(3):
        _, _, P, a = nb['train_phase_classifier'](F, T_lab, lo, hi, seed=s)
        ts.append(nb['crossing'](P, T_lab, T_grid)[0]); accs.append(a)
    allT += ts
    print(f"ends <= {lo}, >= {hi}: validation {np.round(accs, 3)}, crossings {np.round(ts, 2)}")
print(f"all ends and seeds: {min(allT):.2f}-{max(allT):.2f}")

## `exercises/E11_2_galaxy10_params.py`

E11.2  Parameter counts: GalaxyCNN for 3-band, 10-class Galaxy10 images vs the draft's flatten-and-dense design.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E11_2_galaxy10_params.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E11_2_galaxy10_params.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E11.2  Parameter counts: GalaxyCNN for 3-band, 10-class Galaxy10 images vs the draft's flatten-and-dense design.

Answer in the text: first layer 160 -> 448, head 650, other layers 4,640 / 18,496 / 36,928, total 61,162;
draft (69x69 input, conv 32/64/128, flatten 128x8x8 -> 256 -> 10): conv 896 / 18,496 / 73,856, dense 2,097,408 and 2,570,
total 2,193,226 (> 95 % in the first dense layer); global average pooling uses ~36 times fewer parameters.
"""
import sys
from pathlib import Path

import torch.nn as nn

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import listing

ns = listing({'__name__': 'x'}, '01_cnn_architecture.py', 'cnn-arch')
m = ns['GalaxyCNN'](n_classes=10)
m.features[0] = nn.Conv2d(3, 16, 3, padding=1)
per = [sum(p.numel() for p in l.parameters()) for l in m.features if isinstance(l, nn.Conv2d)]
head = sum(p.numel() for p in m.head.parameters())
total = sum(p.numel() for p in m.parameters())
print(f"3-band GalaxyCNN: conv layers {per}, head {head}, total {total:,}")

draft = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Linear(256, 10))
import torch
print("draft output shape for a 69x69 image before flattening:", tuple(draft[:9](torch.zeros(1, 3, 69, 69)).shape))
per_d = [sum(p.numel() for p in l.parameters()) for l in draft if isinstance(l, (nn.Conv2d, nn.Linear))]
tot_d = sum(per_d)
print(f"draft: layers {per_d}, total {tot_d:,}; first dense layer {100 * per_d[3] / tot_d:.1f} %; ratio {tot_d / total:.1f}")

## `exercises/E11_5_rice_arithmetic.py`

E11.5  Arithmetic of the published rice-seed results (Kanchana et al. 2021): overall and balanced accuracy.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E11_5_rice_arithmetic.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E11_5_rice_arithmetic.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E11.5  Arithmetic of the published rice-seed results (Kanchana et al. 2021): overall and balanced accuracy.

Per-variety recall 0.94 / 0.65 / 0.99 / 0.79 for KDML105 / RD23 / RD57 / PTT1; validation counts 65 / 77 / 90 / 80.
Answer in the text: 61, 50, 89, 63 correct = 263 / 312 = 84.3 %; balanced accuracy (0.938 + 0.649 + 0.989 + 0.788)/4 ~ 0.84.
"""
import numpy as np

n = np.array([65, 77, 90, 80]); rec = np.array([0.94, 0.65, 0.99, 0.79])
k = np.round(rec * n).astype(int)
print("correct per variety:", k, "total", k.sum(), "of", n.sum(), f"= {100 * k.sum() / n.sum():.1f} %")
print("recalls from the rounded counts:", np.round(k / n, 3), f"balanced accuracy {np.mean(k / n):.3f}")
print(f"training set 725 + validation 312 = {725 + 312} images")

## `exercises/E11_6_gradcam_regions.py` — โค้ดที่ 11.8

E11.6  Where does the clean CNN look? Share of the Grad-CAM map inside radius 20 px, per class (true class as target).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E11_6_gradcam_regions.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E11_6_gradcam_regions.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E11.6  Where does the clean CNN look? Share of the Grad-CAM map inside radius 20 px, per class (true class as target).

Uses grad_cam of code:gradcam and models['CNN'] (data/CNN.pt from 03_train_evaluate.py).
Answer in the text: the circle is ~31 % of the image; share inside ~86 % (spiral), 68 % (edge-on), 59 % (barred),
but only ~15 % (smooth round) and 22 % (cigar) - less than the area share.
About 20 seconds.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import book_namespace, section

os.chdir(HERE)
ns = book_namespace(trained=True)
exec(section(HERE / '06_gradcam_shortcut.py', '\n# --- code:gradcam', '\ndef add_star'), ns)    # grad_cam only
grad_cam, model, X_test, y_test, CLASSES = ns['grad_cam'], ns['models']['CNN'], ns['X_test'], ns['y_test'], ns['CLASSES']

yy, xx = np.mgrid[:64, :64]
disc = (yy - 31.5)**2 + (xx - 31.5)**2 < 20**2
print(f"area share of the r < 20 disc: {disc.mean():.3f}")
for c in range(5):
    imgs = X_test[y_test == c][:50]
    share = [grad_cam(model, torch.tensor(im[None, None]), c)[disc].sum() for im in imgs]
    print(f"{CLASSES[c]:14s}: share of Grad-CAM inside {np.mean(share):.2f}")

# ตัวอย่างโค้ดที่ไม่ได้รันในโน้ตบุ๊กนี้
เป็นคำสั่งเชลล์ ไฟล์ตั้งค่า โมดูลที่สคริปต์อื่นเรียกใช้ หรือโค้ดสำหรับข้อมูลจริง เปิดดูได้จากแถบไฟล์ของ Colab

| ตัวอย่างโค้ด | ไฟล์ |
|---|---|
| โค้ดที่ 11.6 | `optional/galaxy10_read.py` |

# ข้อมูลจริง (ไม่บังคับ)
สคริปต์สำหรับดาวน์โหลดและอ่านข้อมูลจริงอยู่ใน `ch11_cnn/optional/` วิธีดาวน์โหลดและผลที่ควรได้อยู่ใน `ch11_cnn/README.md` ข้อมูลเหล่านี้ไม่ได้อยู่ใน GitHub เพราะขนาดใหญ่หรือมีเงื่อนไขสัญญาอนุญาต